# code for topic modeling of episode annotations and recall transcripts

### imports

In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import hypertools as hyp
from os.path import join as opj
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer as lemmatizer
from num2words import num2words
from scipy.signal import resample
from scipy.interpolate import interp1d

%matplotlib inline

### set paths

In [2]:
data_dir = '../../data'
annot_dir = opj(data_dir, 'annotations')
transc_dir = opj(data_dir, 'transcriptions', 'manual')
ep_traj_dir = opj(data_dir, 'models', 'episodes', 'trajectories')
rec_traj_dir = opj(data_dir, 'models', 'recalls', 'trajectories')
pickle_dir = opj(data_dir, 'pickles')

### load formatted annotations

In [3]:
atlep1_df = pd.read_csv(opj(annot_dir, 'atlep1.csv'))
atlep2_df = pd.read_csv(opj(annot_dir, 'atlep2.csv'))
arrdev_df = pd.read_csv(opj(annot_dir, 'arrdev.csv'))

### topic modeling parameters

In [4]:
# text preprocessing
stop_words = stopwords.words('english')
extra_stopwords = ['like']
stop_words = stop_words.extend(extra_stopwords)

# combine some multiword phrases, subsitute expletives, etc.
substitutions = {
    # names
    'paper boy': 'paperboy',
    'earnest': 'earn',
    'vanessa': 'van',
    'george-michael': 'georgemichael',
    'flo rida': 'floxrida',
    'david': 'dave',
    # explatives
    'f-word': 'fuck',
    'n-word': 'nigga',
    'racial slur': 'nigga',
    # other words/phrases
    'cause': 'because',
    'low key': 'lowkey',
    'low-key': 'lowkey',
    'parking lot': 'parkinglot',
    'déja': 'deja',
    'weed': 'marijuana',
    'pot': 'marijuana',
    'ex girlfriend': 'exgirlfriend',
    'ex-girlfriend': 'exgirlfriend',
    'ex wife': 'exwife',
    'ex-wife': 'exwife'
}

In [5]:
# t = pd.DataFrame({'test': ['test one', 'test two', 'test three', 'vanessa', 'alfred paper boy', 'paper boy']})
# t['test'].replace({'alfred paper boy': 'paper boy', 'paper boy': 'paperboy'}, regex=True)

In [6]:
# episode_bag = atlep1_df.loc[:,'Narrative details (external events)':'Setting'].apply(
#     lambda x: ', '.join(x.fillna('')), axis=1).values.tolist()
# text = ' '.join(episode_bag)
# ' '.join([lemmatizer().lemmatize(x) for x in text.split()])

In [7]:
# nltk.pos_tag(nltk.word_tokenize(text))

In [8]:
# [x for x in text.split() if x.endswith('ing')]

# for i in text.split():
#     if 
#     n = WordNetLemmatizer().lemmatize(i)
#     v = WordNetLemmatizer().lemmatize(i, 'v')
#     if n != v:
#         print(n, v)

In [9]:
n_topics = 100
episode_wsize = 50    # annotations
recall_wsize = 200    # words

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

## functions

In [36]:
# lower_nopunc = [re.sub("[^\w\s-]+", '', chunk.lower()) for chunk in episode_bag]
# no_acc = [x.replace('é', 'e') for x in lower_nopunc]
# no_digit = [re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), chunk) for chunk in no_acc]
# spaced = [' '.join(x.replace(',', ' ').split()) for x in no_digit]

### for text preprocessing and document formatting

In [45]:
def format_episode_text(textlist):
    """
    standardize annotation text format for modeling
    """
    formatted = []
    for chunk in textlist:
        lower_nopunc = re.sub("[^\w\s-]+", '', chunk.lower())    # remove all punctuation except for dashes
        no_acc = lower_nopunc.replace('é', 'e')    # remove accented character
        no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), no_acc)    # convert digits to words
        spaced = ' '.join(no_digit.replace(',', ' ').split())    # deal with inconsistent whitespace
        formatted.append(spaced)
    
    return formatted

In [185]:
def preprocess_annotations(df_row):
    word_bag = ' '.join(list(df_row.dropna()))
    lower_nopunc = re.sub("[^\w\s-]+", '', word_bag.lower())
    no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), lower_nopunc)

In [205]:
no_digit.replace(*substitutions.items())

TypeError: replace() takes at most 3 arguments (20 given)

In [199]:
pos_tag(['earn'])

[('earn', 'NN')]

In [194]:
# tags = []
# for i in range(len(atlep1_df)):
#     df_row = atlep1_df.loc[:,'Narrative details (external events)':'Setting'].loc[i]
#     rowstring = ' '.join(df_row.dropna().str.lower())
#     tags.extend([j[1] for j in pos_tag(word_tokenize(rowstring))])

In [195]:
word_tokenize(spaced)

['earn',
 'leans',
 'back',
 'in',
 'the',
 'car',
 'seat',
 'looks',
 'at',
 'alfred',
 'paper',
 'boy',
 'and',
 'relaxes',
 'alfred',
 'paper',
 'boy',
 'starts',
 'moving',
 'dancing',
 'to',
 'his',
 'song',
 'on',
 'the',
 'radio',
 'alfred',
 'paper',
 'boy',
 'earn',
 'yes',
 'outdoor',
 'convenience',
 'store',
 'parking',
 'lot']

In [144]:
set(tags)

{'$',
 "''",
 '(',
 ')',
 ',',
 '.',
 ':',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'MD',
 'NN',
 'NNP',
 'NNS',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RP',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WRB',
 '``'}

In [129]:
pos_tag(word_tokenize(' '.join(df_row.dropna().str.lower())))

[('earn', 'NN'),
 ('leans', 'VBZ'),
 ('back', 'RB'),
 ('in', 'IN'),
 ('the', 'DT'),
 ('car', 'NN'),
 ('seat', 'NN'),
 (',', ','),
 ('looks', 'VBZ'),
 ('at', 'IN'),
 ('alfred', 'JJ'),
 ('paper', 'NN'),
 ('boy', 'NN'),
 (',', ','),
 ('and', 'CC'),
 ('relaxes', 'NNS'),
 ('.', '.'),
 ('alfred', 'JJ'),
 ('paper', 'NN'),
 ('boy', 'NN'),
 ('starts', 'VBZ'),
 ('moving', 'VBG'),
 ('/', 'RB'),
 ('dancing', 'VBG'),
 ('to', 'TO'),
 ('his', 'PRP$'),
 ('song', 'NN'),
 ('on', 'IN'),
 ('the', 'DT'),
 ('radio', 'NN'),
 ('.', '.'),
 ('alfred', 'JJ'),
 ('paper', 'NN'),
 ('boy', 'NN'),
 (',', ','),
 ('earn', 'VBP'),
 ('yes', 'UH'),
 ('outdoor', 'JJ'),
 ('convenience', 'NN'),
 ('store', 'NN'),
 ('parking', 'NN'),
 ('lot', 'NN')]

In [178]:
lemmatizer().lemmatize('testing', 'j')

KeyError: 'j'

In [179]:
wordnet.ADJ

'a'

In [176]:
treebank.tagged_words()

[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ...]

In [137]:
# import nltk
nltk.help.upenn_tagset()

$: dollar
    $ -$ --$ A$ C$ HK$ M$ NZ$ S$ U.S.$ US$
'': closing quotation mark
    ' ''
(: opening parenthesis
    ( [ {
): closing parenthesis
    ) ] }
,: comma
    ,
--: dash
    --
.: sentence terminator
    . ! ?
:: colon or ellipsis
    : ; ...
CC: conjunction, coordinating
    & 'n and both but either et for less minus neither nor or plus so
    therefore times v. versus vs. whether yet
CD: numeral, cardinal
    mid-1890 nine-thirty forty-two one-tenth ten million 0.5 one forty-
    seven 1987 twenty '79 zero two 78-degrees eighty-four IX '60s .025
    fifteen 271,124 dozen quintillion DM2,000 ...
DT: determiner
    all an another any both del each either every half la many much nary
    neither no some such that the them these this those
EX: existential there
    there
FW: foreign word
    gemeinschaft hund ich jeux habeas Haementeria Herr K'ang-si vous
    lutihaw alai je jour objets salutaris fille quibusdam pas trop Monte
    terram fiche oui corporis ...
IN: preposition or

In [132]:
from nltk.corpus import wordnet

In [108]:
df_row.str.lower().replace(substitutions, regex=True).values

array(['earn leans back in the car seat, looks at alfred paperboy, and relaxes. alfred paperboy starts moving / dancing to his song on the radio.',
       nan, 'alfred paperboy, earn', 'yes', nan, nan, 'outdoor',
       'convenience store parkinglot'], dtype=object)

In [107]:
df_row.str.lower()

Narrative details (external events)    earn leans back in the car seat, looks at alfr...
Narrative details (internal state)                                                   NaN
Characters on screen                                              alfred paper boy, earn
Music presence                                                                       yes
Speech                                                                               NaN
Character speaking                                                                   NaN
Indoor/outdoor                                                                   outdoor
Setting                                                    convenience store parking lot
Name: 450, dtype: object

In [93]:
atlep1_df.loc[:,'Narrative details (external events)':'Setting'].loc[0].values.tolist()

['Alfred Paper Boy is sitting in his car when a foot kicks off his side view mirror. Alfred Paper Boy turns and exclaims',
 'Alfred Paper Boy is surprised and angry.',
 'Alfred Paper Boy',
 'no',
 'What the?',
 'Alfred Paper Boy',
 'outdoor',
 'convenience store parking lot']

In [86]:
episode_bag = atlep1_df.loc[:,'Narrative details (external events)':'Setting'].apply(
        lambda x: ' '.join(x.dropna()), axis=1).values.tolist()

In [183]:
format_episode_text(episode_bag)[0]

'alfred paper boy is sitting in his car when a foot kicks off his side view mirror alfred paper boy turns and exclaims alfred paper boy is surprised and angry alfred paper boy no what the alfred paper boy outdoor convenience store parking lot'

In [184]:
episode_bag

['Alfred Paper Boy is sitting in his car when a foot kicks off his side view mirror. Alfred Paper Boy turns and exclaims Alfred Paper Boy is surprised and angry. Alfred Paper Boy no What the? Alfred Paper Boy outdoor convenience store parking lot',
 'Alfred Paper Boy and Darius open the doors and get out of the car. A man and woman walk away from the car. Alfred Paper Boy yells after them. Alfred Paper Boy is surprised and angry. Alfred Paper Boy, Darius, Man, Woman no Really man?! Hey man! Hey! Alfred Paper Boy outdoor convenience store parking lot',
 'Alfred Paper Boy steps out of the car. Earn yells his name. Alfred Paper Boy is angry. Earn is concerned. Alfred Paper Boy no Alfred! Earn outdoor convenience store parking lot',
 'Alfred Paper Boy slams the car door and yells after the man and woman, asking for money for his broken mirror. Alfred Paper Boy is angry. Alfred Paper Boy, Earn no Hey man, I\'m gonna need some cash for this mirror, man!" Alfred Paper Boy outdoor Convenience 

In [45]:
def format_episode_text(textlist):
    """
    standardize annotation text format for modeling
    """
    formatted = []
    for chunk in textlist:
        lower_nopunc = re.sub("[^\w\s-]+", '', chunk.lower())    # remove all punctuation except for dashes
        no_acc = lower_nopunc.replace('é', 'e')    # remove accented character
        no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), no_acc)    # convert digits to words
        spaced = ' '.join(no_digit.replace(',', ' ').split())    # deal with inconsistent whitespace
        formatted.append(spaced)
    
    return formatted

In [14]:
# atlep1testpath = opj(transc_dir, 'MD-021919-A-02', 'debugQzo2F:debugV7e7L', 'debugQzo2F:debugV7e7L-recall.txt')
# arrdevtestpath = opj(transc_dir, 'MD-021819-B-02', 'debugNf95q:debugeVCxY', 'debugNf95q:debugeVCxY-recall.txt')
# with open(atlep1testpath, 'r') as f:
#     a1_test = f.read()
# with open(arrdevtestpath, 'r') as f:
#     arrdev_test = f.read()

In [79]:
def get_episode_windows(episode_df, episode_wsize=episode_wsize):
    # create bag of words from annotations for each shot
    episode_bag = episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(
        lambda x: ' '.join(x.dropna()), axis=1).values.tolist()
    formatted = format_episode_text(episode_bag)
    # create sliding windows
    episode_w = []
    for idx, sentence in enumerate(formatted):
        episode_w.append(' '.join(formatted[idx:idx+episode_wsize]))

    return episode_w

In [132]:
def get_recall_windows(transcript, wsize=recall_wsize, grow_beginning=False, shrink_end=True):
    rec_list = transcript.split()
    # create sliding windows
    recall_w = []
    if grow_beginning:
        # first `wsize` windows start with first word and grow until 
        # full window size. Ensures first `wsize` words and last `wsize` 
        # words are in equal number of windows if shrink_end is True
        for ix in range(1, wsize):
            recall_w.append(' '.join(rec_list[0 : ix]))
    if shrink_end:
        # continue appending windows through last word, though last 
        # `wsize` windows will contain < `wsize` words
        word_ix = len(rec_list)
    else:
        # stop shifting sliding window when < `wsize` words remain
        word_ix = len(rec_list) - (wsize - 1)
            
    for ix in range(word_ix):
        recall_w.append(' '.join(rec_list[ix : ix + wsize]))

    return recall_w

### for interpolating episode trajectories

In [ ]:
def find_midpoint_time(df):
    """
    returns list of timepoints at middle of each annotation segment
    """
    # use timestamp of last frame from episode to find midpoint of last annotation
    df_shapes = [atlep1_df.shape, atlep2_df.shape, arrdev_df.shape]
    endframe_times = [1466.0, 1316.52, 1236.6]
    endframe_time = endframe_times[df_shapes.index(df.shape)]

    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        try:
            midpoint_times.append((tpt + df['Onset time'][i+1]) / 2)
        except KeyError:    # handle last annotation
            midpoint_times.append((tpt + endframe_time) / 2)
                    
    return midpoint_times, endframe_time

In [ ]:
def interpolate_episode(traj, df, resolution=1):
    """
    uses linear interpolation to resample episode trajectory timeseries to desired resolution. 
    'resolution' is in units of seconds.
    """
    # get middle timepoint for each annotation
    midpoint_times, endframe_time = find_midpoint_time(df)
    new_traj = np.arange(int(round(endframe_time)), step=resolution)
    interp_func = interp1d(midpoint_times, traj, axis=0, fill_value='extrapolate')
    
    return interp_func(new_traj)

## main topic modeling function

In [ ]:
def fit_and_transform(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
                        corpus=None, resample_shape=None, return_windows=False):
    # handle annotations
    if isinstance(documents, pd.DataFrame):
        windows = get_episode_windows(documents, episode_wsize)
        corpus = windows if not corpus else corpus
        # fit topic model and transform documents
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # interpolate to length of episode (seconds)
        if return_windows:
            return interpolate_episode(traj, documents), windows
        else:
            return interpolate_episode(traj, documents)
        
    # handle recall transcripts
    elif isinstance(documents, str):
        if not corpus:
            raise ValueError("You must pass a training corpus to transform recall transcripts")
        windows = get_recall_windows(documents, recall_wsize)
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # resample to corresponding episode length
        return resample(traj, resample_shape) 

## model episode content, get sliding windows for fitting recall models

In [ ]:
atlep1_traj, atlep1_windows = fit_and_transform(atlep1_df, return_windows=True)
atlep2_traj, atlep2_windows = fit_and_transform(atlep2_df, return_windows=True)
arrdev_traj, arrdev_windows = fit_and_transform(arrdev_df, return_windows=True)

## save episode trajectories

In [ ]:
# np.save(f'{ep_traj_dir}/atlep1_trajectory', atlep1_traj)
# np.save(f'{ep_traj_dir}/atlep2_trajectory', atlep2_traj)
# np.save(f'{ep_traj_dir}/arrdev_trajectory', arrdev_traj)

## load in and model recall transcripts

In [ ]:
recall_trajectories = {
    'atlep1' : [],
    'prediction' : [],
    'delayed' : [],
    'atlep2' : [],
    'arrdev' : []
}

total = sum([len([f for f in files if f.endswith('corrected.wav.txt')]) for r, d, files in os.walk(transc_dir)])
# walk transcription directory structure
currfile = 1
for root, dirs, files in os.walk(transc_dir):
    
    # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
    transcripts = [f for f in files if f.endswith('corrected.wav.txt')]
    for transc in transcripts:

        # assign correct episode windows, corresponding episode trajectory shape, and dict key
        if any('prediction' in t for t in transcripts) or 'delayed' in transc:
            corpus = atlep1_windows
            resample_shape = atlep1_traj.shape[0]
            if 'recall' in transc:
                rectype = 'atlep1'
            elif 'prediction' in transc:
                rectype = 'prediction'
            elif 'delayed' in transc:
                rectype = 'delayed'
            else:
                raise ValueError('Transcript is not a recognized option')
            
        elif '-A-' in root:
            corpus = atlep2_windows
            resample_shape = atlep2_traj.shape[0]
            rectype = 'atlep2'
            
        else:
            corpus = arrdev_windows
            resample_shape = arrdev_traj.shape[0]
            rectype = 'arrdev'
            
        with open(opj(root ,transc), 'r') as f:
            # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
            transcript = ' '.join([line.split(',')[0].lower() for line in f.read().split('\n')])
            
        # fit topic model to episode annotations, 
        print(f'modeling transcript {currfile}/{total}...    {transc}')
        p_traj = fit_and_transform(transcript, resample_shape=resample_shape, corpus=corpus)
        
        recall_trajectories[rectype].append((transc.split('-')[0],p_traj))
        currfile += 1

## save individual trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     for (turkid, traj) in data:
#         np.save(opj(rec_traj_dir, rectype, f'{turkid}.npy'), traj)

## create and save average recall trajectories

In [ ]:
# for rectype, data in recall_trajectories.items():
#     avg_trajectory = np.array([traj for (turkid, traj) in data]).mean(axis=0)
#     np.save(opj(rec_traj_dir, rectype, 'avg_trajectory.npy'), avg_trajectory)